In [1]:
import os
import numpy as np
import pandas as pd
from ultralytics import YOLO
import cv2
from collections import defaultdict
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import yaml

model = YOLO(r'C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt')
base = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train'
csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\semen_analysis_data_Train.csv'
config_path = r'C:\Users\neo62\sperm-ai\bytetrack_custom.yaml'

df_label = pd.read_csv(csv_path)

# Train/Val 분할 (우리가 학습할 때 쓴 것과 동일)
train_ids = ['12','13','15','19','21','24','29','30',
             '35','36','38','47','52','54','60','82']
val_ids   = ['11','14','22','23']

def extract_features(pid):
    """한 참가자의 운동성 특징 추출"""
    video_path = os.path.join(base, pid, f'{pid}.mp4')
    if not os.path.exists(video_path):
        return None

    # 전체 정자 수 N
    cap = cv2.VideoCapture(video_path)
    counts = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret: break
        res = model(frame, verbose=False, conf=0.3)
        counts.append(int((res[0].boxes.cls == 0).sum()))
    cap.release()
    N = int(np.median(counts)) if counts else 0

    # 추적
    cap = cv2.VideoCapture(video_path)
    track_history = defaultdict(list)
    for fidx in range(150):
        ret, frame = cap.read()
        if not ret: break
        res = model.track(frame, persist=True,
                          tracker=config_path,
                          verbose=False, conf=0.3)
        if res[0].boxes.id is not None:
            for box, tid, cls in zip(
                res[0].boxes.xywh.cpu().numpy(),
                res[0].boxes.id.cpu().numpy().astype(int),
                res[0].boxes.cls.cpu().numpy().astype(int)
            ):
                if cls == 0:
                    track_history[tid].append(
                        (fidx, float(box[0]), float(box[1])))
    cap.release()

    # 특징 계산
    speeds, lins, straight_dists = [], [], []
    for tid, pts in track_history.items():
        if len(pts) < 5: continue
        coords = np.array([(cx, cy) for _, cx, cy in pts])
        dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        total_dist = float(np.sum(dists))
        straight_dist = float(np.sqrt(
            (coords[-1][0]-coords[0][0])**2 +
            (coords[-1][1]-coords[0][1])**2))
        avg_speed = total_dist / len(pts)
        linearity = straight_dist / (total_dist + 1e-6)
        speeds.append(avg_speed)
        lins.append(linearity)
        straight_dists.append(straight_dist)

    if not speeds:
        return None

    speeds = np.array(speeds)
    return {
        'pid': pid,
        'N': N,
        'speed_mean':   float(np.mean(speeds)),
        'speed_median': float(np.median(speeds)),
        'speed_75':     float(np.percentile(speeds, 75)),
        'speed_90':     float(np.percentile(speeds, 90)),
        'lin_mean':     float(np.mean(lins)),
        'lin_75':       float(np.percentile(lins, 75)),
        'straight_mean':float(np.mean(straight_dists)),
        'ratio_fast':   float(np.mean(speeds > 1.5)),
        'ratio_medium': float(np.mean((speeds > 0.5) & (speeds <= 1.5))),
        'ratio_slow':   float(np.mean(speeds <= 0.5)),
        'n_tracks':     len(speeds),
    }

# Train 데이터 특징 추출
print("=== Train 참가자 특징 추출 중 ===")
train_features = []
for pid in train_ids:
    print(f"처리 중: 참가자 {pid}...")
    feat = extract_features(pid)
    if feat:
        row = df_label[df_label['ID'] == int(pid)]
        if len(row) > 0:
            feat['prog'] = float(row['Progressive motility (%)'].values[0])
            feat['non_prog'] = float(row['Non progressive sperm motility (%)'].values[0])
            feat['immotile'] = float(row['Immotile sperm (%)'].values[0])
            train_features.append(feat)

print(f"\nTrain 데이터 추출 완료: {len(train_features)}명")
train_df = pd.DataFrame(train_features)
train_df.to_csv(r'C:\Users\neo62\sperm-ai\outputs\train_features.csv', index=False)
print("저장 완료: outputs/train_features.csv")

=== Train 참가자 특징 추출 중 ===
처리 중: 참가자 12...
처리 중: 참가자 13...
처리 중: 참가자 15...
처리 중: 참가자 19...
처리 중: 참가자 21...
처리 중: 참가자 24...
처리 중: 참가자 29...
처리 중: 참가자 30...
처리 중: 참가자 35...
처리 중: 참가자 36...
처리 중: 참가자 38...
처리 중: 참가자 47...
처리 중: 참가자 52...
처리 중: 참가자 54...
처리 중: 참가자 60...
처리 중: 참가자 82...

Train 데이터 추출 완료: 16명
저장 완료: outputs/train_features.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
import warnings
warnings.filterwarnings('ignore')

# 학습 데이터 로드
train_df = pd.read_csv(r'C:\Users\neo62\sperm-ai\outputs\train_features.csv')

# 입력 특징
feature_cols = [
    'speed_mean', 'speed_median', 'speed_75', 'speed_90',
    'lin_mean', 'lin_75', 'straight_mean',
    'ratio_fast', 'ratio_medium', 'ratio_slow', 'n_tracks'
]

X_train = train_df[feature_cols].values
y_train = train_df[['prog', 'non_prog', 'immotile']].values

# 스케일링 + 학습
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model_reg = MultiOutputRegressor(Ridge(alpha=1.0))
model_reg.fit(X_train_scaled, y_train)

# Train 성능 확인
y_pred_train = model_reg.predict(X_train_scaled)
mae_train = np.mean(np.abs(y_pred_train - y_train), axis=0)
print("=== Train 성능 (참고용) ===")
print(f"전진 운동성 MAE:   {mae_train[0]:.1f}%p")
print(f"비전진 운동성 MAE: {mae_train[1]:.1f}%p")
print(f"비운동성 MAE:      {mae_train[2]:.1f}%p")

# Val 참가자 특징 추출
print("\n=== Val 참가자 특징 추출 중 ===")
val_ids = ['11', '14', '22', '23']
val_features = []

for pid in val_ids:
    print(f"처리 중: 참가자 {pid}...")
    feat = extract_features(pid)
    if feat:
        row = df_label[df_label['ID'] == int(pid)]
        if len(row) > 0:
            feat['prog'] = float(row['Progressive motility (%)'].values[0])
            feat['non_prog'] = float(row['Non progressive sperm motility (%)'].values[0])
            feat['immotile'] = float(row['Immotile sperm (%)'].values[0])
            val_features.append(feat)

val_df = pd.DataFrame(val_features)

# Val 예측
X_val = val_df[feature_cols].values
X_val_scaled = scaler.transform(X_val)
y_pred_val = model_reg.predict(X_val_scaled)

# 결과 출력
print("\n=== Val 검증 결과 ===")
print(f"{'참가자':<6} {'AI전진':>6} {'실제전진':>8} {'AI비전진':>8} {'실제비전진':>10} {'AI비운동':>8} {'실제비운동':>10}")
print("-" * 60)

y_val_true = val_df[['prog', 'non_prog', 'immotile']].values
for i, pid in enumerate(val_df['pid'].values):
    ai = y_pred_val[i]
    real = y_val_true[i]
    print(f"{pid:<6} {ai[0]:>6.1f}% {real[0]:>7.1f}% "
          f"{ai[1]:>7.1f}% {real[1]:>9.1f}% "
          f"{ai[2]:>7.1f}% {real[2]:>9.1f}%")

mae_val = np.mean(np.abs(y_pred_val - y_val_true), axis=0)
print("-" * 60)
print(f"\n=== Val MAE (진짜 성능) ===")
print(f"전진 운동성:   {mae_val[0]:.1f}%p")
print(f"비전진 운동성: {mae_val[1]:.1f}%p")
print(f"비운동성:      {mae_val[2]:.1f}%p")
print(f"전체 평균:     {np.mean(mae_val):.1f}%p")

=== Train 성능 (참고용) ===
전진 운동성 MAE:   6.1%p
비전진 운동성 MAE: 5.2%p
비운동성 MAE:      5.4%p

=== Val 참가자 특징 추출 중 ===
처리 중: 참가자 11...
처리 중: 참가자 14...
처리 중: 참가자 22...
처리 중: 참가자 23...

=== Val 검증 결과 ===
참가자      AI전진     실제전진    AI비전진      실제비전진    AI비운동      실제비운동
------------------------------------------------------------
11       40.8%    11.0%    29.6%      17.0%    29.6%      72.0%
14       26.0%    41.0%    39.6%      43.0%    34.5%      16.0%
22       39.6%    56.0%    37.9%      25.0%    22.5%      19.0%
23       30.5%    18.0%    34.9%      34.0%    34.5%      48.0%
------------------------------------------------------------

=== Val MAE (진짜 성능) ===
전진 운동성:   18.4%p
비전진 운동성: 7.5%p
비운동성:      19.5%p
전체 평균:     15.1%p
